In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

data_dir = Path("/content/drive/MyDrive/MVJ_Internship/Data")

print("Files found in Data folder:")
for file in sorted(data_dir.iterdir()):
    print(file.name)

Files found in Data folder:
online_retail_II_csv(2009-2010).csv
online_retail_II_csv(2010-2011).csv


In [5]:
file_1 = data_dir / "online_retail_II_csv(2009-2010).csv"
file_2 = data_dir / "online_retail_II_csv(2010-2011).csv"

print("2009-10 file exists:", file_1.exists())
print("2010-2011 file exists:", file_2.exists())

2009-10 file exists: True
2010-2011 file exists: True


In [7]:
#load both csvs as a dataframe for pandas

df_1 = pd.read_csv(
    file_1,
    low_memory=False,
    dtype={"Invoice": "string", "StockCode": "string"},
    encoding="ISO-8859-1"
)

df_2 = pd.read_csv(
    file_2,
    low_memory=False,
    dtype={"Invoice": "string", "StockCode": "string"},
    encoding="ISO-8859-1"
)

print("2009-2010 shape:", df_1.shape)
print("2010-2011 shape:", df_2.shape)

display(df_1.head())
display(df_2.head())

2009-2010 shape: (525461, 8)
2010-2011 shape: (541910, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,01-12-09,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,01-12-09,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,01-12-09,1.25,13085.0,United Kingdom


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,1-12-10,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,1-12-10,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,1-12-10,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,1-12-10,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,1-12-10,3.39,17850.0,United Kingdom


In [8]:
#combine the data sets
df = pd.concat(
    [df_1, df_2],
    ignore_index=True
)

print("Combined shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())

Combined shape: (1067371, 8)

Column names:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,01-12-09,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,01-12-09,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,01-12-09,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,01-12-09,1.25,13085.0,United Kingdom


In [9]:
#inspect and prepare the columns
print("Data types:")
display(df.dtypes.to_frame("Data Type"))

print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

df["Quantity"] = pd.to_numeric(
    df["Quantity"],
    errors="coerce"
)

df["Price"] = pd.to_numeric(
    df["Price"],
    errors="coerce"
)

print("\nUpdated data types:")
display(df[["InvoiceDate", "Quantity", "Price"]].dtypes.to_frame("Data Type"))

print("\nDate range:")
print("Earliest:", df["InvoiceDate"].min())
print("Latest:", df["InvoiceDate"].max())

Data types:


,Data Type
Invoice,string[python]
StockCode,string[python]
Description,object
Quantity,int64
InvoiceDate,object
Price,float64
Customer ID,float64
Country,object



Missing values:


,Missing Values
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0



Updated data types:


/tmp/ipykernel_3211/503016983.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["InvoiceDate"] = pd.to_datetime(


,Data Type
InvoiceDate,datetime64[ns]
Quantity,int64
Price,float64



Date range:
Earliest: 2009-01-12 00:00:00
Latest: 2011-12-10 00:00:00


In [10]:
#filter the sales records
invoice_text = df["Invoice"].fillna("").str.strip()

valid_rows = (
    df["InvoiceDate"].notna()
    & df["Quantity"].gt(0)
    & df["Price"].gt(0)
    & ~invoice_text.str.upper().str.startswith("C")
)

sales = df.loc[valid_rows].copy()

sales["Revenue"] = sales["Quantity"] * sales["Price"]

print("Original rows:", len(df))
print("Included sales rows:", len(sales))
print("Excluded rows:", len(df) - len(sales))

display(sales.head())

Original rows: 1067371
Included sales rows: 1041670
Excluded rows: 25701


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-01-12,6.95,13085.0,United Kingdom,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-01-12,6.75,13085.0,United Kingdom,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-01-12,6.75,13085.0,United Kingdom,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-01-12,2.10,13085.0,United Kingdom,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-01-12,1.25,13085.0,United Kingdom,30.0


In [11]:
#calculate the overall KPIs
total_revenue = sales["Revenue"].sum()
total_orders = sales["Invoice"].nunique()

average_order_value = (
    total_revenue / total_orders
    if total_orders > 0
    else np.nan
)

overall_kpis = pd.DataFrame([{
    "total_revenue": round(total_revenue, 2),
    "total_orders": total_orders,
    "average_order_value": round(average_order_value, 2),
    "transaction_rows": len(sales),
    "unique_products": sales["StockCode"].nunique()
}])

display(overall_kpis)

,total_revenue,total_orders,average_order_value,transaction_rows,unique_products
0,20972594.57,40077,523.31,1041670,4917


In [12]:
#calculate monthly revenue and MoM growth
sales["Month"] = sales["InvoiceDate"].dt.to_period("M")

monthly_kpis = (
    sales.groupby("Month")
    .agg(
        revenue=("Revenue", "sum"),
        orders=("Invoice", "nunique")
    )
    .sort_index()
)

monthly_kpis["average_order_value"] = (
    monthly_kpis["revenue"]
    / monthly_kpis["orders"].replace(0, np.nan)
)

monthly_kpis["mom_growth_percent"] = (
    monthly_kpis["revenue"].pct_change(fill_method=None) * 100
)

monthly_kpis = monthly_kpis.reset_index()
monthly_kpis["Month"] = monthly_kpis["Month"].astype(str)

for column in ["revenue", "average_order_value", "mom_growth_percent"]:
    monthly_kpis[column] = monthly_kpis[column].round(2)

display(monthly_kpis)

,Month,revenue,orders,average_order_value,mom_growth_percent
0,2009-01,54513.50,119,458.10,NaN
1,2009-02,63352.51,115,550.89,16.21
2,2009-03,74037.91,124,597.08,16.87
3,2009-04,40732.92,89,457.67,-44.98
4,2009-05,9803.05,30,326.77,-75.93
5,2009-06,24613.64,74,332.62,151.08
6,2009-07,45083.35,101,446.37,83.16
7,2009-08,49517.23,109,454.29,9.83
8,2009-09,40616.09,88,461.55,-17.98
9,2009-10,44442.11,92,483.07,9.42


In [13]:
#calculate repeat-customer rate
customer_sales = sales.loc[
    sales["Customer ID"].notna()
].copy()

customer_order_counts = (
    customer_sales.groupby("Customer ID")["Invoice"].nunique()
)

identified_customers = len(customer_order_counts)
repeat_customers = int((customer_order_counts >= 2).sum())

repeat_rate = (
    repeat_customers / identified_customers * 100
    if identified_customers > 0
    else np.nan
)

repeat_kpi = pd.DataFrame([{
    "identified_customers": identified_customers,
    "repeat_customers": repeat_customers,
    "repeat_customer_rate_percent": round(repeat_rate, 2)
}])

display(repeat_kpi)

,identified_customers,repeat_customers,repeat_customer_rate_percent
0,5878,4255,72.39


In [14]:
#find the top 10 products by revenue
top_products = (
    sales.groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .agg(
        total_revenue=("Revenue", "sum"),
        units_sold=("Quantity", "sum"),
        orders=("Invoice", "nunique")
    )
    .sort_values("total_revenue", ascending=False)
    .head(10)
    .reset_index()
)

top_products["total_revenue"] = top_products["total_revenue"].round(2)

display(top_products)

,StockCode,Description,total_revenue,units_sold,orders
0,22423,REGENCY CAKESTAND 3 TIER,344563.25,27577,3918
1,M,Manual,340716.28,10050,784
2,DOT,DOTCOM POSTAGE,322657.48,1436,1415
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,262931.16,96086,5356
4,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
5,47566,PARTY BUNTING,149187.05,28378,2674
6,85099B,JUMBO BAG RED RETROSPOT,148823.92,78860,3245
7,84879,ASSORTED COLOUR BIRD ORNAMENT,132187.92,81809,2807
8,POST,POSTAGE,127597.42,5461,1851
9,22086,PAPER CHAIN KIT 50'S CHRISTMAS,123141.54,36581,2018


In [15]:
#compare with SQL results
sql_summary_path = Path(
    "/content/drive/MyDrive/MVJ_Internship/Project1/day8_kpi/kpi_summary.csv"
)

if sql_summary_path.exists():
    sql_summary = pd.read_csv(sql_summary_path)
    display(sql_summary)
else:
    print("SQL summary not found. Check the Day 8 output folder.")

,total_revenue,total_orders,average_order_value,transaction_rows,unique_products
0,20972594.57,40077,523.31,1041670,4917


In [16]:
#save the pandas outputs
output_dir = Path(
    "/content/drive/MyDrive/MVJ_Internship/Project1/day10_pandas"
)

output_dir.mkdir(parents=True, exist_ok=True)

overall_kpis.to_csv(output_dir / "pandas_kpi_summary.csv", index=False)
monthly_kpis.to_csv(output_dir / "pandas_monthly_kpis.csv", index=False)
repeat_kpi.to_csv(output_dir / "pandas_repeat_customer_kpi.csv", index=False)
top_products.to_csv(output_dir / "pandas_top_10_products.csv", index=False)

print("Saved files:")
for file in sorted(output_dir.glob("*.csv")):
    print(file.name)

Saved files:
pandas_kpi_summary.csv
pandas_monthly_kpis.csv
pandas_repeat_customer_kpi.csv
pandas_top_10_products.csv
